<a href="https://colab.research.google.com/github/ARCHITHAOJHA/ReinforcementLearning/blob/main/LAB_10_2170.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assignment1: Write Python code for Bellman Optimality and Value Iteration in a Deterministic
Grid World

In [1]:
# Assignment 1: Bellman Optimality and Value Iteration in a Deterministic Grid World

# Grid layout:
# S0 | WALL
# S1 | G
#
# Convention: actions that hit a boundary or the wall leave the agent in its
# current state and give reward -1. Entering G gives +10; G is terminal and V(G)=0.
# Value iteration uses synchronous updates (all new values use the previous values).

from pprint import pprint

states = ["S0", "S1", "G"]
non_terminal_states = ["S0", "S1"]
actions = ["up", "down", "left", "right"]
gamma = 0.9
theta = 1e-10
wall = "WALL"
terminal_state = "G"

# Transition mapping: state -> action -> (next_state, immediate_reward)
transitions = {
    "S0": {
        "up":    ("S0", -1),  # boundary
        "down":  ("S1", -1),
        "left":  ("S0", -1),  # boundary
        "right": ("S0", -1),  # wall
    },
    "S1": {
        "up":    ("S0", -1),
        "down":  ("S1", -1),  # boundary
        "left":  ("S1", -1),  # boundary
        "right": ("G", +10),
    }
}

print("1. MDP INSPECTION")
print("Grid: S0 | WALL")
print("      S1 | G")
print("States:", states)
print("Non-terminal states:", non_terminal_states)
print("Actions:", actions)
print("Discount factor gamma:", gamma)
print("Rewards: normal move = -1; wall/boundary collision = -1; entering G = +10")
print("Terminal condition: V(G) = 0; no actions are taken from G.")
print("\nTransition mapping:")
pprint(transitions)

print("\n2. TRANSITION VERIFICATION")
pair_count = 0
for state in non_terminal_states:
    for action in actions:
        next_state, reward = transitions[state][action]
        print(f"state={state:2s}, action={action:5s}, next_state={next_state:2s}, reward={reward:+}")
        pair_count += 1
assert pair_count == len(non_terminal_states) * len(actions)
assert all(action in transitions[state] for state in non_terminal_states for action in actions)
print(f"Verified: {pair_count} of {len(non_terminal_states) * len(actions)} state-action pairs have transitions.")

print("\n3. INITIAL STATE VALUES")
V = {"S0": 0.0, "S1": 0.0, "G": 0.0}
print(V)

print("\n4-5. SYNCHRONOUS VALUE ITERATION")
history = []
iteration = 0
while True:
    iteration += 1
    old_V = V.copy()
    new_V = old_V.copy()  # preserve terminal value
    for state in non_terminal_states:
        q_values = []
        for action in actions:
            next_state, reward = transitions[state][action]
            q_values.append(reward + gamma * old_V[next_state])
        new_V[state] = max(q_values)
    delta = max(abs(new_V[s] - old_V[s]) for s in non_terminal_states)
    V = new_V
    history.append((iteration, V["S0"], V["S1"], V["G"], delta))
    print(f"Iteration {iteration:2d}: V(S0)={V['S0']:.10f}, V(S1)={V['S1']:.10f}, "
          f"V(G)={V['G']:.1f}, delta={delta:.10f}")
    if delta < theta:
        break

print(f"\nConverged after {iteration} iterations with theta={theta:g}.")
print("\nIteration-wise table:")
print(f"{'k':>3} {'V(S0)':>14} {'V(S1)':>14} {'V(G)':>8} {'Delta':>14}")
for row in history:
    k, v0, v1, vg, d = row
    print(f"{k:3d} {v0:14.8f} {v1:14.8f} {vg:8.2f} {d:14.8f}")

print("\n6. FINAL ACTION VALUES Q*(s,a)")
Q = {}
for state in non_terminal_states:
    Q[state] = {}
    for action in actions:
        next_state, reward = transitions[state][action]
        Q[state][action] = reward + gamma * V[next_state]
    print(f"{state}:")
    for action in actions:
        print(f"  Q*({state}, {action}) = {Q[state][action]:.10f}")

print("\n7. OPTIMAL POLICY (all tied maximizing actions)")
policy = {}
for state in non_terminal_states:
    best_q = max(Q[state].values())
    best_actions = [a for a, q in Q[state].items() if abs(q - best_q) < 1e-8]
    policy[state] = best_actions
    print(f"pi*({state}) = {best_actions} (value {best_q:.10f})")

print("\n8. BELLMAN OPTIMALITY VERIFICATION")
for state in non_terminal_states:
    max_q = max(Q[state].values())
    print(f"{state}: V*({state})={V[state]:.10f}; max_a Q*({state},a)={max_q:.10f}; "
          f"equal={abs(V[state] - max_q) < 1e-8}")
    assert abs(V[state] - max_q) < 1e-8

print("\n9. INTERPRETATION")
print("- In S1, moving right to G is optimal: Q = +10 + 0.9*V(G) = +10.")
print("- In S0, moving down to S1 is optimal: Q = -1 + 0.9*10 = 8.")
print("- The goal reward raises the value of states from which the goal can be reached.")
print("- gamma=0.9 discounts future rewards: the +10 received after moving from S0 via S1")
print("  contributes 0.9*10=9 before the -1 cost of entering S1, giving V*(S0)=8.")
print("- The environment is deterministic. The prompt mentions a stochastic policy, but")
print("  value iteration uses a max over actions to compute the optimal deterministic policy.")

1. MDP INSPECTION
Grid: S0 | WALL
      S1 | G
States: ['S0', 'S1', 'G']
Non-terminal states: ['S0', 'S1']
Actions: ['up', 'down', 'left', 'right']
Discount factor gamma: 0.9
Rewards: normal move = -1; wall/boundary collision = -1; entering G = +10
Terminal condition: V(G) = 0; no actions are taken from G.

Transition mapping:
{'S0': {'down': ('S1', -1),
        'left': ('S0', -1),
        'right': ('S0', -1),
        'up': ('S0', -1)},
 'S1': {'down': ('S1', -1),
        'left': ('S1', -1),
        'right': ('G', 10),
        'up': ('S0', -1)}}

2. TRANSITION VERIFICATION
state=S0, action=up   , next_state=S0, reward=-1
state=S0, action=down , next_state=S1, reward=-1
state=S0, action=left , next_state=S0, reward=-1
state=S0, action=right, next_state=S0, reward=-1
state=S1, action=up   , next_state=S0, reward=-1
state=S1, action=down , next_state=S1, reward=-1
state=S1, action=left , next_state=S1, reward=-1
state=S1, action=right, next_state=G , reward=+10
Verified: 8 of 8 state-acti